# MI-GNAN: step-by-step training

Select the **gnan-clean** kernel. Set `DATASET` in the first cell and run
sections **1–7**, through saving: they support every dataset in the registry.

Available datasets: `mutagenicity`, `nci1`, `proteins`, `ptc_mr`, `bareg1`,
`bareg2`, `crippen`, `qm9_mu`, `qm9_alpha`, `qm9_homo`.



In [17]:
from pathlib import Path
import sys
import json

cwd = Path.cwd().resolve()
package = next((p for parent in (cwd, *cwd.parents)
                for p in (parent / "github" / "mignan", parent / "mignan", parent)
                if all((p / name).is_file() for name in ("__init__.py", "settings.py", "model.py"))), None)
if package is None:
    raise FileNotFoundError("Open the notebook from the Clara or mignan directory")
if str(package.parent) not in sys.path:
    sys.path.insert(0, str(package.parent))

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from mignan.settings import DATASETS, MOTIF_SETTINGS, MODEL_SETTINGS, TRAIN_SETTINGS
from mignan.data import load_dataset, split_graphs, Preprocessor, loader
from mignan.motifs import build_topology
from mignan.model import MIGNAN
from mignan.training import fit, evaluate, seed_everything, save_model, load_model

torch.set_num_threads(2)
DATASET = "qm9_mu"
MAX_GRAPHS = None  # None: full dataset; an integer: reproducible subset.
SUBSET_SEED = 9182
SPLIT_SEED = 2027
config = DATASETS[DATASET]
print(config["name"], "|", config["task"], "| target:", config["target"])
print("Available device:", "cuda" if torch.cuda.is_available() else "cpu")


QM9-mu | regression | target: mu
Available device: cuda


## 1. Data and feature schemas

`settings.py` declares the loader, target, and schema for each dataset.
One-hot groups become categorical variables with lookup tables; numerical
variables use MLPs. PROTEINS and QM9 combine both types. For QM9, the requested
target is selected without changing the features. Classes are encoded in
`preprocessing.classes`.


In [18]:
graphs = load_dataset(DATASET, max_graphs=MAX_GRAPHS, seed=SUBSET_SEED)
train_idx, val_idx, test_idx = split_graphs(graphs, task=config["task"], seed=SPLIT_SEED)
schema = config["features"]
display(pd.DataFrame([{"feature": f.name, "kind": f.kind, "columns": f.columns,
                       "categories": f.categories} for f in schema]))
print("Training / validation / test graphs:", len(train_idx), len(val_idx), len(test_idx))
print("Raw features per node:", graphs[0].x.shape[1])


,feature,kind,columns,categories
0,atom,one_hot,"(0, 1, 2, 3, 4)","(H, C, N, O, F)"
1,atomic_number,numerical,"(5,)",()
2,aromatic,categorical,"(6,)","(no, yes)"
3,sp,categorical,"(7,)","(no, yes)"
4,sp2,categorical,"(8,)","(no, yes)"
5,sp3,categorical,"(9,)","(no, yes)"
6,num_hydrogens,numerical,"(10,)",()


Training / validation / test graphs: 91581 19625 19625
Raw features per node: 11


## 2. Motifs and caching

Mining uses only training graphs. The frozen vocabulary is applied to all graphs.
Dataset-specific options override the shared defaults: QM9 uses motifs of
orders 3–4 and at most 5000 training graphs for mining.

The compressed archive stores all candidate counts and null-replicate totals, independently of FDR. Occurrences are computed lazily for explanations.
With the same graphs, split, and options, it is loaded without rerunning mining.
Use `force_recompute=True` to force recomputation.

For a quick check, set `MAX_GRAPHS` in the first cell and, below,
`motif_options.update(sizes=(3,), vocabulary_mode="all", null_replicates=0)`.
This mode checks the code; it is not an enrichment benchmark.


In [19]:
motif_options = {**MOTIF_SETTINGS, **config.get("motifs", {})}
motifs = build_topology(graphs, train_idx, **motif_options, force_recompute=False)
print("Loaded from cache" if motifs.cache_hit else "Computed and saved")
print(motifs.cache_path)
display(motifs.vocabulary[["column", "motif", "size", "observed", "q_value"]])
display(pd.DataFrame(motifs.counts[0], columns=motifs.vocabulary.motif).head())


print("selected motifs", motifs.counts[0].shape[1])

Loaded from cache
/home/antonio/Clara/github/mignan/motif_counts/candidates_669734fd92b3801ba694b96f7278c0c7fddf155062e9dd06f234cfc637e8bb75.npz


,column,motif,size,observed,q_value
0,0,M1,3,3063894,0.008269
1,1,M2,4,1613569,0.008269
2,2,M3,5,307765,0.008269
3,3,M4,5,5919031,0.008269
4,4,M5,5,4492197,0.008269
5,5,M6,4,4098409,0.008269
6,6,M7,5,966,0.008269


motif,M1,M2,M3,M4,M5,M6,M7
0,6,4,1,0,0,0,0
1,3,3,1,0,0,0,0
2,3,3,1,0,0,0,0
3,3,3,1,0,0,0,0
4,3,3,1,0,0,0,0


selected motifs 7


In [11]:
R = motifs.null_counts.shape[0]
print("Repliche effettive:", R, "| p-value minimo:", 1 / (R + 1))

display(motifs.vocabulary[
    ["motif", "observed", "null_mean", "null_std",
     "z_score", "p_value", "q_value"]
])

Repliche effettive: 500 | p-value minimo: 0.001996007984031936


,motif,observed,null_mean,null_std,z_score,p_value,q_value
0,M1,3063894,2846670.138,841.923706,258.008963,0.001996,0.008269
1,M2,1613569,1314808.780,1195.698394,249.862525,0.001996,0.008269
2,M3,307765,213525.584,444.275604,212.119268,0.001996,0.008269
3,M4,5919031,4748544.330,5940.076908,197.049077,0.001996,0.008269
4,M5,4492197,3949819.216,3615.604381,150.010269,0.001996,0.008269
5,M6,4098409,3815968.820,2394.488989,117.954261,0.001996,0.008269
6,M7,966,463.956,21.609164,23.232921,0.001996,0.008269


In [12]:
# A second call with the same inputs reuses the counts.
cache_check = build_topology(graphs, train_idx, **motif_options)
assert cache_check.cache_hit
assert all(np.array_equal(a, b) for a, b in zip(motifs.counts, cache_check.counts))
print("Cache verified")

Cache verified


## 3. Preparing inputs

Numerical features are standardized on the training set; categorical features
become integer indices. Topological counts undergo `log1p` followed by
standardization on the training set. Regression targets are also standardized.
These input scalers weight every **node** equally; the subsequent function
centering weights every **graph** equally.


In [13]:
preprocessing = Preprocessor.fit(graphs, motifs.counts, train_idx, schema, config["task"])
records = preprocessing.transform(graphs, motifs.counts)
print("Logical variables:", len(schema), "| Motifs:", len(motifs.vocabulary))
if config["task"] == "classification":
    print("Classes:", preprocessing.classes)
else:
    print("Target:", config["target"], "| training mean/scale:",
          preprocessing.target_mean, preprocessing.target_scale)
print("Example transformed inputs:", records[0][0][:5])


Logical variables: 7 | Motifs: 7
Target: mu | training mean/scale: 2.6750539510736506 1.5051298412999043
Example transformed inputs: tensor([[ 1.0000,  0.8460,  0.0000,  0.0000,  0.0000,  0.0000,  4.0497],
        [ 0.0000, -0.9579,  0.0000,  0.0000,  0.0000,  0.0000, -0.5946],
        [ 0.0000, -0.9579,  0.0000,  0.0000,  0.0000,  0.0000, -0.5946],
        [ 0.0000, -0.9579,  0.0000,  0.0000,  0.0000,  0.0000, -0.5946],
        [ 0.0000, -0.9579,  0.0000,  0.0000,  0.0000,  0.0000, -0.5946]])


## 4. Creating MI-GNAN

Four independent function banks represent feature effects, motif effects,
feature factors, and motif factors. Products are computed **at the same node**
and then summed over the graph. The rank is 1 for each logit.


In [14]:
train_options = {**TRAIN_SETTINGS, **config.get("training", {})}
# Adjust epochs, batch_size, lr, etc. here before creating the model.
seed_everything(train_options["seed"])
model = MIGNAN(schema, num_motifs=len(motifs.vocabulary),
               outputs=preprocessing.outputs, **MODEL_SETTINGS)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))


MIGNAN(
  (feature_main): ShapeBank(
    (shapes): ModuleList(
      (0): Embedding(5, 1)
      (1): Sequential(
        (0): Linear(in_features=1, out_features=16, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=16, out_features=1, bias=True)
      )
      (2-5): 4 x Embedding(2, 1)
      (6): Sequential(
        (0): Linear(in_features=1, out_features=16, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=16, out_features=1, bias=True)
      )
    )
  )
  (topology_main): ShapeBank(
    (shapes): ModuleList(
      (0-6): 7 x Sequential(
        (0): Linear(in_features=1, out_features=16, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=16, out_features=1, bias=True)
      )
    )
  )
  (feature_factor): ShapeBank(
    (shapes): ModuleList(
      (0): Embedding(5, 1)
      (1): Sequential(
        (0): Linear(in_featu

## 5. Sparse training

`fit` trains the model just created and restores the best validation checkpoint.
Binary classification uses BCE and AUROC for selection; multiclass classification
uses cross-entropy. Regression uses MSE on the standardized target and selects
with RMSE in original units. Epochs, patience, and batch size follow the dataset.
The total loss is `task_loss + lambda_sparse * penalty`.
The penalty sums the absolute values of the four centered function banks,
averaging first over nodes within each graph and then over graphs. With multiple
logits, it also averages over outputs.

Centering means are recomputed over the entire training set before training and
after each epoch, with dropout and gradients disabled. They remain fixed during
each epoch's updates. `lambda_sparse=0` disables regularization; the default is
a starting point, not a tuned value.


In [15]:
result = fit(model, records, train_idx, val_idx, preprocessing, **train_options)
print("Selected epoch:", result.best_epoch)
display(result.history.tail())

Epoch   1 | task=0.7348 | sparsity=0.8973 | val_rmse=1.2792
Epoch  10 | task=0.6199 | sparsity=2.2132 | val_rmse=1.4959
Epoch  20 | task=0.6123 | sparsity=2.6246 | val_rmse=1.5309
Epoch  30 | task=0.5963 | sparsity=2.8000 | val_rmse=1.4689
Epoch  40 | task=0.5927 | sparsity=2.8622 | val_rmse=1.5063
Epoch  50 | task=0.5868 | sparsity=2.9604 | val_rmse=1.5710
Selected epoch: 2


,epoch,train_task,train_sparsity,train_loss,val_loss,val_rmse,val_mae,val_r2
47,48,0.584915,2.949514,0.587864,0.985698,1.494328,1.199219,0.006288
48,49,0.584135,2.954439,0.587090,0.982726,1.492074,1.195303,0.009284
49,50,0.586827,2.960408,0.589788,1.089433,1.570993,1.283778,-0.098290
50,51,0.595909,2.975918,0.598885,1.037011,1.532730,1.242631,-0.045442
51,52,0.604845,3.005123,0.607850,0.997669,1.503375,1.211343,-0.005780


## 6. Test evaluation

The test set is evaluated after checkpoint selection. Classification reports
labels and probabilities; regression predictions, RMSE, and MAE are expressed
in the original target units.


In [16]:
metrics, predictions = evaluate(
    model, loader(records, test_idx, batch_size=train_options["batch_size"]), preprocessing
)
display(pd.Series(metrics, name="test"))
prediction_table = pd.DataFrame({
    "graph": predictions["graph_ids"],
    "target": predictions["targets"],
    "prediction": predictions["predictions"],
})
if config["task"] == "classification":
    for column, label in enumerate(preprocessing.classes):
        prediction_table[f"P(class={label:g})"] = predictions["probabilities"][:, column]
display(prediction_table.head())


loss    0.699462
rmse    1.258798
mae     0.919028
r2      0.295866
Name: test, dtype: float64

,graph,target,prediction
0,6,0.0000,1.522494
1,7,1.5258,1.231587
2,11,3.7286,2.149547
3,24,0.0023,5.246089
4,37,1.7341,2.204018


## 7. Saving and loading

Save only the model, exact split, and a compact reproducibility manifest.
The checkpoint includes the feature schema, scalers, centering, and motif vocabulary.
Histories and per-graph predictions are not written to disk.

Use `explore.ipynb` to load this run or a winning model from `tune.py`.


In [ ]:
output_dir = package / "results" / DATASET
output_dir.mkdir(parents=True, exist_ok=True)
save_model(output_dir / "model.pt", model, preprocessing, motifs.vocabulary)
np.savez_compressed(output_dir / "split.npz", train=train_idx, validation=val_idx, test=test_idx)
from mignan.tune import experiment_metadata

metadata = experiment_metadata(
    DATASET, graphs, max_graphs=MAX_GRAPHS, subset_seed=SUBSET_SEED,
    split_seed=SPLIT_SEED, motif_options=motif_options, cache_path=motifs.cache_path,
    model_options=MODEL_SETTINGS,
    training_options={**train_options, "device": str(model.intercept.device)},
)
metadata["models"] = [{
    "seed": train_options["seed"], "checkpoint": "model.pt",
    "best_epoch": result.best_epoch, "validation_score": result.best_validation,
    "test_metrics": {k: v if np.isfinite(v) else None for k, v in metrics.items()},
}]
(output_dir / "run.json").write_text(json.dumps(metadata, indent=2, allow_nan=False) + "\n")

restored, restored_preprocessing, vocabulary = load_model(output_dir / "model.pt")
_, restored_predictions = evaluate(
    restored, loader(records, test_idx, batch_size=train_options["batch_size"]),
    restored_preprocessing,
)
np.testing.assert_allclose(
    restored_predictions["predictions"], predictions["predictions"], rtol=1e-4, atol=1e-4
)
print("Checkpoint reloaded and predictions verified:", output_dir / "model.pt")
